<a href="https://colab.research.google.com/github/mbakos95/aki-sentinel/blob/main/notebooks/01_cohort_definition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AKI Sentinel - Cohort Definition

## Goal

Build the initial ICU cohort for the AKI prediction experiment.

At this stage, we only use:

- patients
- admissions
- icustays

The purpose is to understand the relationship between patients,
hospital admissions, and ICU stays before adding clinical features
or defining the AKI outcome.

In [1]:
# Import necessary libraries for data manipulation and path handling.
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Define root and sub-directories for data storage.
DATA_ROOT= Path("/content/drive/MyDrive/Early Acute Kidney Injury Prediction + Production Monitoring/data")
HOSP_DIR = DATA_ROOT /"hosp"
ICU_DIR = DATA_ROOT /"icu"

In [3]:
# Load patient, admission, and ICU stay data from gzipped CSV files.
patients = pd.read_csv(
    HOSP_DIR / "patients.csv.gz",
    compression="gzip"
)
admission = pd.read_csv(
    HOSP_DIR / "admissions.csv.gz",
    compression="gzip"
)
icustays = pd.read_csv(
    ICU_DIR / "icustays.csv.gz",
    compression="gzip"
)

In [4]:
# Print the dimensions (rows, columns) of the loaded DataFrames.
print("patients", patients.shape)
print("admission", admission.shape)
print("icustays", icustays.shape)

patients (364627, 6)
admission (546028, 16)
icustays (94458, 8)


In [5]:
# Display the column names for each of the loaded DataFrames.
print("patients")
print(patients.columns.tolist())
print("\nadmission")
print(admission.columns.tolist())
print("\nicustays")
print(icustays.columns.tolist())

patients
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']

admission
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']

icustays
['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']


In [6]:
# Display the first few rows of the 'patients' DataFrame.
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [7]:
# Display the first few rows of the 'admission' DataFrame.
admission.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


In [8]:
# Display the first few rows of the 'icustays' DataFrame.
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


In [9]:
# Print the number of rows and unique IDs for patients, admissions, and ICU stays.
print("PATIENTS")
print("Rows:", len(patients))
print("Unique subject_id:", patients["subject_id"].nunique())

print("\nADMISSIONS")
print("Rows:", len(admission))
print("Unique hadm_id:", admission["hadm_id"].nunique())
print("Unique subject_id:", admission["subject_id"].nunique())

print("\nICUSTAYS")
print("Rows:", len(icustays))
print("Unique stay_id:", icustays["stay_id"].nunique())
print("Unique hadm_id:", icustays["hadm_id"].nunique())
print("Unique subject_id:", icustays["subject_id"].nunique())

PATIENTS
Rows: 364627
Unique subject_id: 364627

ADMISSIONS
Rows: 546028
Unique hadm_id: 546028
Unique subject_id: 223452

ICUSTAYS
Rows: 94458
Unique stay_id: 94458
Unique hadm_id: 85242
Unique subject_id: 65366


In [10]:
# Calculate and display the top 10 patients with the most admissions.
admissions_per_patient = (
    admission
    .groupby("subject_id")
    .size()
    .sort_values(ascending=False)
)

admissions_per_patient.head(10)

,0
subject_id,
15496609,238
15464144,185
10714009,163
16662316,142
14394983,138
15229574,130
11582633,105
17011846,104
13475033,103


In [11]:
# Calculate and display the top 10 admissions with the most ICU stays.
icu_stays_per_admission = (
    icustays
    .groupby("hadm_id")
    .size()
    .sort_values(ascending=False)
)

icu_stays_per_admission.head(10)

,0
hadm_id,
29633749,10
20673627,9
23344494,7
24673862,7
25178923,7
20084622,7
28682905,6
24949837,6
25332191,6


In [12]:
# Merge icustays with patient demographic data.
cohort = icustays.merge(
    patients,
    on="subject_id",
    how="left",
    validate="many_to_one"
)

In [13]:
# Merge the combined cohort with admission details.
cohort = cohort.merge(
    admission,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one",
    suffixes=["_icu", "_adm"]
)

In [14]:
# Print the shape of the merged cohort and display its head.
print("Cohort shape:", cohort.shape)

cohort.head()

Cohort shape: (94458, 27)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,...,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,...,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,F,86,...,P26QQ4,EMERGENCY ROOM,REHAB,Medicare,English,WIDOWED,WHITE,2150-11-02 11:41:00,2150-11-02 19:37:00,0
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,F,73,...,P06OTX,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,2189-06-27 06:25:00,2189-06-27 08:42:00,0
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,...,P3610N,EMERGENCY ROOM,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,2157-11-18 17:38:00,2157-11-19 01:24:00,0
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,F,55,...,P276OU,PHYSICIAN REFERRAL,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,NaN,NaN,0


In [15]:
# Compare original ICU stay count with the merged cohort's row count and unique stay IDs.
print("Original ICU stays:", len(icustays))
print("Rows after joins:", len(cohort))
print("Unique stay_id:", cohort["stay_id"].nunique())

Original ICU stays: 94458
Rows after joins: 94458
Unique stay_id: 94458


In [16]:
# Convert relevant columns to datetime objects.
datetime_columns = [
    "intime",
    "outtime",
    "admittime",
    "dischtime"
]

for col in datetime_columns:
    cohort[col] = pd.to_datetime(cohort[col])

In [17]:
# Calculate the length of ICU stay in hours.
cohort["icu_los_hours"] = (
    cohort["outtime"] - cohort["intime"]
).dt.total_seconds() / 3600

In [18]:
# Display descriptive statistics for the ICU length of stay.
cohort["icu_los_hours"].describe()

,icu_los_hours
count,94444.000000
mean,87.120596
std,129.659365
min,0.030000
25%,26.309097
50%,47.175556
75%,92.701806
max,5433.673889


In [19]:
# Define prediction time (12 hours after ICU admission) and prediction end (24 hours after prediction time).
cohort["prediction_time"] = (
    cohort["intime"] + pd.Timedelta(hours=12)
)

cohort["prediction_end"] = (
    cohort["prediction_time"] + pd.Timedelta(hours=24)
)

In [20]:
# Calculate the patient's age at the time of ICU admission.
cohort["age"] = (
    cohort["anchor_age"]
    + cohort["intime"].dt.year
    - cohort["anchor_year"]
)

In [21]:
# Display key identifier and time-related columns for the first 10 entries of the cohort.
cohort[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "outtime",
        "icu_los_hours",
        "age",
        "prediction_time",
        "prediction_end"
    ]
].head(10)

,subject_id,hadm_id,stay_id,intime,outtime,icu_los_hours,age,prediction_time,prediction_end
0,10000032,29079034,39553978,2180-07-23 14:00:00,2180-07-23 23:50:47,9.846389,52,2180-07-24 02:00:00,2180-07-25 02:00:00
1,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,93.438056,86,2150-11-03 07:37:00,2150-11-04 07:37:00
2,10000980,26913865,39765666,2189-06-27 08:42:00,2189-06-27 20:38:27,11.940833,76,2189-06-27 20:42:00,2189-06-28 20:42:00
3,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,26.832778,55,2157-11-21 07:18:02,2157-11-22 07:18:02
4,10001217,27703517,34592300,2157-12-19 15:42:24,2157-12-20 14:27:41,22.754722,55,2157-12-20 03:42:24,2157-12-21 03:42:24
5,10001725,25563031,31205490,2110-04-11 15:52:22,2110-04-12 23:59:56,32.126111,46,2110-04-12 03:52:22,2110-04-13 03:52:22
6,10001843,26133978,39698942,2134-12-05 18:50:03,2134-12-06 14:38:26,19.806389,76,2134-12-06 06:50:03,2134-12-07 06:50:03
7,10001884,26184834,37510196,2131-01-11 04:20:05,2131-01-20 08:27:30,220.123611,77,2131-01-11 16:20:05,2131-01-12 16:20:05
8,10002013,23581541,39060235,2160-05-18 10:00:53,2160-05-19 17:33:33,31.544444,57,2160-05-18 22:00:53,2160-05-19 22:00:53
9,10002114,27793700,34672098,2162-02-17 23:30:00,2162-02-20 21:16:27,69.774167,56,2162-02-18 11:30:00,2162-02-19 11:30:00


In [22]:
# Filter the cohort to include only adult patients (age 18 or older).
adult_cohort = cohort[
    cohort["age"] >= 18
].copy()

In [23]:
# Print the number of rows before and after applying the adult age filter.
print("Before adult filter:", len(cohort))
print("After adult filter:", len(adult_cohort))
print("Removed:", len(cohort) - len(adult_cohort))

Before adult filter: 94458
After adult filter: 94458
Removed: 0


In [24]:
# Filter the adult cohort to include only ICU stays where the patient is still in the ICU at the prediction time.
eligible_cohort = adult_cohort[
    adult_cohort["outtime"] >= adult_cohort["prediction_time"]
].copy()

In [25]:
# Create and display a DataFrame summarizing the cohort filtering steps and counts.
cohort_flow = pd.DataFrame({
    "Step": [
        "All ICU stays",
        "Adults (age >= 18)",
        "Still in ICU at 12h prediction time"
    ],
    "N": [
        len(cohort),
        len(adult_cohort),
        len(eligible_cohort)
    ]
})

cohort_flow

,Step,N
0,All ICU stays,94458
1,Adults (age >= 18),94458
2,Still in ICU at 12h prediction time,90467


In [26]:
# Print the final counts for eligible ICU stays, unique patients, and unique admissions.
print(f"Final eligible ICU stays: {len(eligible_cohort):,}")
print(f"Unique patients: {eligible_cohort['subject_id'].nunique():,}")
print(f"Unique admissions: {eligible_cohort['hadm_id'].nunique():,}")

Final eligible ICU stays: 90,467
Unique patients: 63,487
Unique admissions: 82,422
